# 📊 UnthAI Annotation Dashboard
This notebook provides a real-time snapshot of the annotation progress and label distributions in `annotation_part_1.csv`.

In [ ]:
%pip install seaborn

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Settings
FILE_PATH = "annotation_part_1.csv"
CATEGORIES = ['food', 'service', 'place', 'delivery', 'price', 'treatment']

if not os.path.exists(FILE_PATH):
    print(f"❌ File {FILE_PATH} not found.")
else:
    df = pd.read_csv(FILE_PATH, keep_default_na=False)
    print(f"✅ Loaded {len(df)} rows from {FILE_PATH}")

## 1. Overall Progress

In [ ]:
# A row is considered labeled if 'out_of_scope' is True or False
labeled_mask = df['out_of_scope'].isin(['True', 'False', True, False])
total_labeled = labeled_mask.sum()
total_rows = len(df)
progress = (total_labeled / total_rows) * 100

print(f"📌 Total Rows: {total_rows}")
print(f"✅ Total Labeled: {total_labeled}")
print(f"⏳ Remaining: {total_rows - total_labeled}")
print(f"📈 Progress: {progress:.2f}%")

# Visualization
plt.figure(figsize=(6, 6))
plt.pie([total_labeled, total_rows - total_labeled], labels=['Labeled', 'Remaining'], autopct='%1.1f%%', colors=['#4CAF50', '#f44336'], startangle=140)
plt.title('Annotation Progress')
plt.show()

## 2. Intent Distribution per Category

In [ ]:
# Pre-processing: handle empty strings as 'None'
for cat in CATEGORIES:
    df[cat] = df[cat].apply(lambda x: 'None' if str(x).strip() == '' or str(x) == 'nan' else str(x))

# Plotting
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for i, cat in enumerate(CATEGORIES):
    counts = df[labeled_mask][cat].value_counts()
    sns.barplot(x=counts.index, y=counts.values, ax=axes[i], palette='viridis')
    axes[i].set_title(f'Distribution: {cat.upper()}')
    axes[i].set_ylabel('Count')
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. High-Level Insights

In [ ]:
oos_counts = df[labeled_mask]['out_of_scope'].astype(str).value_counts()
print("--- Out of Scope Distribution ---")
print(oos_counts)

# Multi-label stats
def count_labels(row):
    return sum(1 for cat in CATEGORIES if row[cat] != 'None')

df['label_count'] = df[labeled_mask].apply(count_labels, axis=1)
multi_label_counts = df['label_count'].value_counts().sort_index()

print("\n--- Multi-Label Engagement ---")
for count, freq in multi_label_counts.items():
    print(f"{int(count)} categories tagged: {freq} comments")